In [5]:
import pandas

In [6]:
import requests

In [7]:
import torch 

In [8]:
import requests 
import pandas as pd 
import numpy as np
import json
import time 
import os 
from pathlib import Path

In [19]:
os.makedirs("data/raw", exist_ok=True)
COUNTRIES = [
    "KEN", "TZA", "UGA", "ETH", "RWA",       # East Africa
    "NGA", "GHA", "CIV", "SEN",               # West Africa
    "EGY", "MAR", "TUN",                      # North Africa
    "ZAF", "BWA", "ZMB",                      # Southern Africa
    "USA", "GBR", "DEU", "FRA", "JPN",        # Major advanced
    "CHN", "IND", "BRA", "MEX", "IDN",        # Large EM (IDN not TUR twice)
    "TUR", "THA", "MYS", "PHL", "VNM",        # Asia EM
    "SAU", "ARE", "NOR", "AUS", "CAN",        # Commodity exporters ← comma here
    "POL", "CZE", "HUN", "ROU",               # Eastern Europe
    "ARG", "CHL", "COL", "PER",               # Latin America
    "KOR", "SGP", "PAK", "BGD",              # Other Asia ← comma here
    "ISL", "NZL", "CRI",                      # Small open economies
]
INDICATORS= {
    "NY.GDP.MKTP.KD.ZG": "gdp_growth",
    "FP.CPI.TOTL.ZG": "inflation",
    "SL.UEM.TOTL.ZS": "unemployment",
    "FR.INR.LEND": "lending_rate",
    "NE.TRD.GNFS.ZS": "trade_opennes",
    "BN.CAB.XOKA.GD.ZS": "govt_debt",
    "NY.GDP.PCAP.KD.ZG": "gdp_per_capita_growth"
}

START_YEAR= 1995
END_YEAR=2023
# RAW_PATH = Path(r"data/raw/wb_raw.csv")
# PROCESSED_PATH=Path(r"data/processed/panel.csv")
# TENSOR_PATH= Path(r"data/processed/tensor.npy")
# META_PATH = Path(r"data/processed/metadata.json")

In [20]:
def fetch_indicator(code, name, countries ,start_year,end_year):
    country_str=";".join(countries)
    url=(
        f"https://api.worldbank.org/v2/country/{country_str}/indicator/{code}?date={start_year}:{end_year}&format=json&per_page=10000"
    )
    resp=requests.get(url,timeout=30)
    resp.raise_for_status()
    payload=resp.json()
        
    if not isinstance(payload,list):
        print(f"Unexpected response for {code}: {payload}")
        return pd.DataFrame()
    if len(payload) < 2 or not payload[1]:
        print(f"Unexpected response for {code}: {payload}")
        print(f" Raw response: {payload[0]}")
        return pd.DataFrame()

    records=[
        {
            "country_iso3":item["countryiso3code"],
            "country_name": item["country"]["value"],
            "year": int(item["date"]),
            name: item["value"]
        }
        for item in payload[1]
    ]
    return pd.DataFrame(records)

panel=None

for code,name in INDICATORS.items():
    print(f"Fetching: {name}...")
    df= fetch_indicator(code,name,COUNTRIES,START_YEAR,END_YEAR)
    panel= df if panel is None else panel.merge(
        df[["country_iso3","year",name]],
        on=["country_iso3","year"],
        how="outer"

    )
    time.sleep(0.5)

panel = panel.sort_values(["country_iso3","year"]).reset_index(drop=True)

panel.to_csv("data/raw/wb_panel_raw.csv", index=False)

print(f"\nShape: {panel.shape}")
print(f"Countries: {panel['country_iso3'].nunique()}")
print(f"Years: {panel["year"].min()} - {panel['year'].max()}")

print("\n --- Missing Value Per Column-----")
for col in list(INDICATORS.values()):
    n= panel[col].isna().sum()
    pct= panel[col].isna().mean() * 100
    print(f" {col:<28} {n:>5} missing ({pct:.1f}%)")

print("\n--- Kenya rows(all years)--------")
print(panel[panel["country_iso3"]=="KEN"].to_string(index=False))
print("\nSaved")


Fetching: gdp_growth...
Fetching: inflation...
Fetching: unemployment...
Fetching: lending_rate...
Fetching: trade_opennes...
Fetching: govt_debt...
Fetching: gdp_per_capita_growth...

Shape: (1450, 10)
Countries: 50
Years: 1995 - 2023

 --- Missing Value Per Column-----
 gdp_growth                       0 missing (0.0%)
 inflation                       38 missing (2.6%)
 unemployment                     0 missing (0.0%)
 lending_rate                   421 missing (29.0%)
 trade_opennes                   51 missing (3.5%)
 govt_debt                       62 missing (4.3%)
 gdp_per_capita_growth            0 missing (0.0%)

--- Kenya rows(all years)--------
country_iso3 country_name  year  gdp_growth  inflation  unemployment  lending_rate  trade_opennes  govt_debt  gdp_per_capita_growth
         KEN        Kenya  1995    4.406217   1.554328         2.848     28.795833      71.745742 -17.446216               1.483737
         KEN        Kenya  1996    4.146839   8.864087         2.773   